# CE541E08 — Unit 4 · Day 35 — Visualisation with Pandas and Matplotlib
| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 4 — The Pandas Library |
| **Session** | Day 35 of 45 |
| **Topics** | time series · scatter · boxplot · bar chart · trend line |
---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 35"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Visualisation in Hydrology

Effective visualisation is essential for communicating hydrological findings. Today we build four standard chart types used in hydrology reports:

| Chart type | Use case |
|---|---|
| **Line plot** | Time series — flow over months or years |
| **Scatter plot** | Rainfall vs streamflow relationship |
| **Box plot** | Monthly flow distribution across many years |
| **Bar chart** | Annual rainfall with trend line |

---
## Code Block 1 — Time Series: Monthly Streamflow

### What this code does

We plot 5 years of monthly streamflow as a line chart with a filled area under the curve and a mean reference line.

### Why each step is taken

**`ax.fill_between(x, 0, y, alpha=0.2)`:**
Fills the area between the line and the x-axis with a semi-transparent blue. This visually emphasises the magnitude of flow — more area = more water. `alpha=0.2` makes it transparent so the line remains readable.

**`ax.axhline(mean, ...)`:**
Horizontal reference line at the long-term mean. Any bar or line above this represents above-average conditions.

**`fig, ax = plt.subplots()`:**
Creates a Figure and Axes separately. Using `ax.plot()` instead of `plt.plot()` is preferred for multi-panel figures and gives more control over each subplot.

### Algorithm

```
1. 60-month DataFrame (2020-2024) with DatetimeIndex

2. ax.plot(df.index, df['Flow_m3s']) → line chart
   ax.fill_between(x, 0, y) → filled area under line

3. ax.axhline(mean) → horizontal mean reference

4. Labels, title, legend, tight_layout, show
```

### Expected output

*(A time series line plot — verify the red dashed mean line is visible)*

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
dates = pd.date_range('2020-01-01','2024-12-31',freq='ME')
base  = np.array([45,38,28,22,35,234,456,389,198,89,62,50])
flow  = np.tile(base,5) + np.random.normal(0,30,60)
df    = pd.DataFrame({'Flow_m3s':flow}, index=dates)

fig, ax = plt.subplots(figsize=(12,4))

# Line chart with filled area under the curve
ax.plot(df.index, df['Flow_m3s'], 'b-', linewidth=1, label='Monthly mean flow')
ax.fill_between(df.index, 0, df['Flow_m3s'], alpha=0.2, color='blue')

# Mean reference line
ax.axhline(df['Flow_m3s'].mean(), color='red', linestyle='--',
           label=f"Mean ({df['Flow_m3s'].mean():.0f} m3/s)")

ax.set_xlabel('Date')
ax.set_ylabel('Flow (m3/s)')
ax.set_title('Monthly Mean Streamflow — KRS Station 2020-24')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Mean: {df['Flow_m3s'].mean():.1f}, Max: {df['Flow_m3s'].max():.1f} m3/s")

### 🔁 Try this

Change `linewidth=1` to `linewidth=3` and `alpha=0.2` to `alpha=0.5`.

How does the plot change? Which combination is clearest for a report?

---
## Code Block 2 — Scatter Plot: Rainfall vs Streamflow

### What this code does

We plot 120 daily observations as a scatter plot coloured by flow magnitude, and compute the Pearson correlation coefficient.

### Why each step is taken

**`ax.scatter(..., c=df['Flow_m3s'], cmap='YlOrRd')`:**
Colours each point by its flow value using the Yellow-Orange-Red colourmap. This adds a third variable (flow magnitude) to a 2-D scatter plot — high flows appear in red, low flows in yellow.

**`plt.colorbar(sc, ...)`:**
Adds a colourbar legend showing the mapping from colour to flow value. Without this, the colour encoding is meaningless to the reader.

**`df['Rainfall_mm'].corr(df['Flow_m3s'])`:**
Pearson correlation coefficient r. r near +1 = strong positive linear relationship (more rain → more flow). The value is embedded in the plot title so it is always visible.

### Algorithm

```
1. 120-day DataFrame: Rainfall_mm and Flow_m3s
   Flow is simulated as rain*8 + noise

2. scatter(x=Rainfall, y=Flow, c=Flow, cmap='YlOrRd')
   → each point coloured by its flow value

3. colorbar → legend for the colour scale

4. corr = Rainfall.corr(Flow) → Pearson r
   Title includes r value
```

### Expected output

*(Scatter plot — verify that higher rainfall points are in red/orange)*

```
Correlation r = 0.xxx
```

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(5)
n    = 120
rain = np.round(np.random.exponential(30, n), 1)
flow = np.round(rain*8 + np.random.normal(50,30,n), 1)
df   = pd.DataFrame({'Rainfall_mm':rain, 'Flow_m3s':np.maximum(flow,0)})

fig, ax = plt.subplots(figsize=(7,5))

# Scatter plot coloured by flow magnitude
sc = ax.scatter(df['Rainfall_mm'], df['Flow_m3s'],
                c=df['Flow_m3s'], cmap='YlOrRd',
                alpha=0.7, edgecolors='grey', linewidths=0.3)

plt.colorbar(sc, ax=ax, label='Flow (m3/s)')

corr = df['Rainfall_mm'].corr(df['Flow_m3s'])
ax.set_xlabel('Daily Rainfall (mm)')
ax.set_ylabel('Streamflow (m3/s)')
ax.set_title(f'Rainfall vs Streamflow  (r={corr:.3f})')
plt.tight_layout()
plt.show()
print(f"Correlation r = {corr:.3f}")

### 🔁 Try this

Change the simulation to `flow = rain*4 + noise` (weaker rainfall-runoff relationship).

- Does the scatter plot look different?
- Does the correlation r decrease?
- What does a lower r mean physically?

---
## Code Block 3 — Box Plot: Monthly Flow Distribution

### What this code does

We plot the distribution of daily flow for each calendar month using box plots — showing median, quartiles, and outliers. The monsoon months are coloured differently from the dry months.

### Why each step is taken

**`[df[df['Month']==m]['Flow_m3s'].values for m in range(1,13)]`:**
Creates a list of 12 arrays — one per calendar month — each containing all daily values for that month across 10 years. This is the input format `plt.boxplot` expects: a list of arrays.

**`patch_artist=True`:**
Enables filled box patches. Without this, boxes are outlined only (not filled), which makes colour assignment impossible.

**Colouring monsoon months:**
`colors = ['lightblue']*5 + ['steelblue']*4 + ['lightblue']*3` gives steelblue (darker) to months 6-9 (Jun-Sep) and lightblue to all other months. This immediately distinguishes the monsoon from the dry season visually.

### Algorithm

```
1. 10-year daily DataFrame (2015-2024)

2. monthly_data = [values for each calendar month across all years]
   → list of 12 arrays

3. ax.boxplot(monthly_data, labels=month_names, patch_artist=True)
   → 12 box plots side by side

4. Colour monsoon months darker
```

### Expected output

*(Box plot — verify that monsoon months Jun-Sep are darker coloured and have higher median values)*

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
dates = pd.date_range('2015-01-01','2024-12-31',freq='D')
base  = np.where((dates.month>=6)&(dates.month<=9), 350, 65)
flow  = np.round(np.maximum(base+np.random.normal(0,base*0.3,len(dates)),5),1)
df    = pd.DataFrame({'Flow_m3s':flow,'Month':dates.month,'Year':dates.year},index=dates)

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# List of 12 arrays: one per calendar month (all values across all years)
monthly_data = [df[df['Month']==m]['Flow_m3s'].values for m in range(1,13)]

fig, ax = plt.subplots(figsize=(12,5))

# boxplot: one box per month; patch_artist=True allows filled coloured boxes
bp = ax.boxplot(monthly_data, labels=month_names, patch_artist=True)

# Colour monsoon months (6-9) darker
colors = ['lightblue']*5 + ['steelblue']*4 + ['lightblue']*3
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_xlabel('Month')
ax.set_ylabel('Flow (m3/s)')
ax.set_title('Monthly Streamflow Distribution — 10-Year Record (2015-2024)')
plt.tight_layout()
plt.show()

### 🔁 Try this

Change `base = np.where(...)` so that October and November are also included in the high-flow season (simulate a dam that releases water in Oct-Nov).

How does the box plot change?

---
## Code Block 4 — Bar Chart with Trend Line

### What this code does

We plot 20 years of annual rainfall as a bar chart, colouring above-mean years in blue and below-mean years in salmon, and add a linear trend line computed with `np.polyfit`.

### Why each step is taken

**Conditional colour list:**
`['steelblue' if v>=mean else 'salmon' for v in df['Annual_mm']]` creates a list of colours — blue for wet years, red for dry years. This makes the departure from normal immediately visible without needing the reader to compare each bar to the mean line.

**`np.polyfit(x, y, 1)`:**
Fits a 1st-degree polynomial (straight line) to the data using least squares. Returns `[slope, intercept]`. `np.poly1d(z)` converts to a callable function: `p(year)` gives the fitted value for any year.

**`z[0]` — slope:**
The slope of the trend line in mm per year. Negative slope = declining rainfall. Shown in the legend label so the reader sees the rate of change.

### Algorithm

```
1. 20-year annual rainfall with slight downward trend

2. Bar chart: colour each bar blue or salmon based on
   whether it is above or below the long-term mean

3. np.polyfit(years, rainfall, 1) → [slope, intercept]
   np.poly1d(z)(years) → fitted values for trend line

4. Plot axhline for mean and trend line
5. Print slope in mm/year
```

### Expected output

*(Bar chart — above-average years in blue, below in salmon, red trend line)*

```
Trend: -5.0 mm per year (approximate)
```

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
years       = list(range(2005, 2025))
annual_rain = [950-i*1.2+np.random.normal(0,80) for i,yr in enumerate(years)]
df = pd.DataFrame({'Year':years,'Annual_mm':annual_rain})

fig, ax = plt.subplots(figsize=(10,5))

# Conditional colour: blue for above-mean, salmon for below-mean years
colors = ['steelblue' if v>=df['Annual_mm'].mean() else 'salmon'
          for v in df['Annual_mm']]
ax.bar(df['Year'], df['Annual_mm'], color=colors, edgecolor='white')

# Mean reference line
ax.axhline(df['Annual_mm'].mean(), color='navy', linestyle='--', linewidth=1.5,
           label=f"Mean ({df['Annual_mm'].mean():.0f} mm)")

# Linear trend line using np.polyfit
z = np.polyfit(df['Year'], df['Annual_mm'], 1)  # [slope, intercept]
p = np.poly1d(z)   # callable function: p(year) = slope*year + intercept
ax.plot(df['Year'], p(df['Year']), 'r-', linewidth=2,
        label=f"Trend ({z[0]:+.1f} mm/yr)")

ax.set_xlabel('Year')
ax.set_ylabel('Annual Rainfall (mm)')
ax.set_title('Annual Rainfall Trend — Bengaluru 2005-2024')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Trend: {z[0]:+.1f} mm per year")

### 🔁 Try this

Change the trend from `-1.2` mm/yr to `+2.0` mm/yr (increasing rainfall).

- Does the trend line slope upward now?
- Does the label in the legend show a positive sign?

---
## Session Summary — Visualisation

| Chart type | Code | Use case |
|---|---|---|
| Time series | `ax.plot(df.index, df['col'])` | Flow/rainfall over time |
| Filled area | `ax.fill_between(x, 0, y, alpha=0.2)` | Emphasise magnitude |
| Mean line | `ax.axhline(mean, color, linestyle)` | Reference level |
| Scatter | `ax.scatter(x, y, c=z, cmap='YlOrRd')` | Rainfall vs flow |
| Colourbar | `plt.colorbar(sc, label=...)` | Legend for colour scale |
| Box plot | `ax.boxplot(list_of_arrays, patch_artist=True)` | Monthly distribution |
| Bar chart | `ax.bar(x, y, color=colour_list)` | Annual values |
| Trend line | `np.polyfit(x,y,1)` → `np.poly1d(z)(x)` | Linear regression |

---
## Day 35 Assignment

10-year daily streamflow (2015-2024):

1. Plot annual mean flow as a bar chart — colour above-mean years blue, below-mean years salmon
2. Add a trend line using `np.polyfit`
3. Print the trend direction and rate (mm/yr or m³/s per year)

### ▶ Assignment cell

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

np.random.seed(0)
dates=pd.date_range('2015-01-01','2024-12-31',freq='D')
base=np.where((dates.month>=6)&(dates.month<=9),350,65)
flow=np.round(np.maximum(base+np.random.normal(0,base*0.3,len(dates)),5),1)
df=pd.DataFrame({'Flow_m3s':flow,'Month':dates.month,'Year':dates.year},index=dates)

# 1. Annual mean per year
annual_mean = df.groupby('Year')['Flow_m3s'].mean()

# 2. Bar chart
fig,ax=plt.subplots(figsize=(10,4))
ax.bar(annual_mean.index, annual_mean.values, color='steelblue')
ax.axhline(annual_mean.mean(), color='red', linestyle='--', label='Overall mean')
ax.set_title('Annual Mean Streamflow'); ax.legend(); plt.tight_layout(); plt.show()
print(f"Overall mean: {annual_mean.mean():.1f} m3/s")

---
- [ ] Run all cells — verify outputs
- [ ] Complete the assignment cell
- [ ] Upload: `Unit4_Pandas/CE541E08_U4_Day35.ipynb`
- [ ] Commit: `Day 35 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*